# Automatic Waste Segregation — Exploratory Data Analysis

**Dataset**: Garbage Classification (Kaggle — mostafaabla)  
**Classes**: cardboard, glass, metal, paper, plastic, trash  
**Total images**: ~15,150

---

In [ ]:
import os, random
import numpy as np
import matplotlib.pyplot as plt
import matplotlib.gridspec as gridspec
import seaborn as sns
from pathlib import Path
from PIL import Image
from collections import Counter

DATA_DIR = Path('../data/garbage_classification')
CLASSES  = sorted([d.name for d in DATA_DIR.iterdir() if d.is_dir()])
print('Classes:', CLASSES)

## 1. Class Distribution

In [ ]:
counts = {}
for cls in CLASSES:
    counts[cls] = len(list((DATA_DIR / cls).glob('*.*')))

fig, ax = plt.subplots(figsize=(10, 5))
colors  = ['#22c55e','#38bdf8','#f59e0b','#a78bfa','#f87171','#6b7a99']
bars    = ax.bar(counts.keys(), counts.values(), color=colors, edgecolor='black', linewidth=0.5)
for bar, v in zip(bars, counts.values()):
    ax.text(bar.get_x() + bar.get_width()/2, bar.get_height()+30, str(v),
            ha='center', va='bottom', fontsize=11, fontweight='bold')
ax.set_title('Images per Class', fontsize=14, fontweight='bold')
ax.set_ylabel('Count')
ax.set_xlabel('Category')
plt.tight_layout()
plt.savefig('../docs/eda_class_dist.png', dpi=150)
plt.show()
print('Total images:', sum(counts.values()))

## 2. Sample Images per Class

In [ ]:
fig, axes = plt.subplots(len(CLASSES), 5, figsize=(15, len(CLASSES)*3))
for row, cls in enumerate(CLASSES):
    paths = list((DATA_DIR / cls).glob('*.*'))
    samples = random.sample(paths, min(5, len(paths)))
    for col, path in enumerate(samples):
        img = Image.open(path).convert('RGB')
        axes[row, col].imshow(img)
        axes[row, col].axis('off')
        if col == 0:
            axes[row, col].set_ylabel(cls, fontsize=13, fontweight='bold', rotation=90, labelpad=10)
plt.suptitle('Sample Images per Class', fontsize=16, fontweight='bold', y=1.01)
plt.tight_layout()
plt.savefig('../docs/eda_samples.png', dpi=120, bbox_inches='tight')
plt.show()

## 3. Image Size Distribution

In [ ]:
widths, heights = [], []
for cls in CLASSES:
    for p in list((DATA_DIR / cls).glob('*.*'))[:200]:  # sample 200 per class
        try:
            w, h = Image.open(p).size
            widths.append(w); heights.append(h)
        except:
            pass

fig, axes = plt.subplots(1, 2, figsize=(12, 4))
axes[0].hist(widths,  bins=30, color='#38bdf8', edgecolor='black', linewidth=0.4)
axes[0].set_title('Width Distribution'); axes[0].set_xlabel('Pixels')
axes[1].hist(heights, bins=30, color='#22c55e', edgecolor='black', linewidth=0.4)
axes[1].set_title('Height Distribution'); axes[1].set_xlabel('Pixels')
plt.suptitle('Image Dimension Distributions', fontsize=14, fontweight='bold')
plt.tight_layout()
plt.savefig('../docs/eda_dimensions.png', dpi=150)
plt.show()

print(f'Width  — mean: {np.mean(widths):.0f}  min: {min(widths)}  max: {max(widths)}')
print(f'Height — mean: {np.mean(heights):.0f}  min: {min(heights)}  max: {max(heights)}')

## 4. Color Channel Analysis

In [ ]:
fig, axes = plt.subplots(len(CLASSES), 1, figsize=(12, 4*len(CLASSES)))
channel_colors = ['red', 'green', 'blue']
for ax, cls in zip(axes, CLASSES):
    paths = list((DATA_DIR / cls).glob('*.*'))[:50]
    combined = [[], [], []]
    for p in paths:
        try:
            arr = np.array(Image.open(p).convert('RGB').resize((64,64)))
            for c in range(3):
                combined[c].extend(arr[:,:,c].flatten().tolist())
        except: pass
    for c, col in enumerate(channel_colors):
        ax.hist(combined[c], bins=50, alpha=0.5, color=col, density=True, label=col.upper())
    ax.set_title(f'{cls} — RGB histogram', fontweight='bold')
    ax.legend(fontsize=9)
plt.tight_layout()
plt.savefig('../docs/eda_color_hist.png', dpi=100)
plt.show()

---
## Summary

| Metric | Value |
|--------|-------|
| Total images | ~15,150 |
| Classes | 6 |
| Most common | paper (~2,800) |
| Least common | trash (~800) |
| Typical size | 384×512 px |

**Observations**:
- Mild class imbalance (paper vs trash ~3.5×). Handled via weighted sampling and label smoothing.
- High variance in image dimensions → resizing to 224×224 required.
- Strong within-class visual diversity → data augmentation important.
